# TeacherLM Report Charts

This notebook generates the charts for the TeacherLM report from the retrieval benchmark and the current Course Builder data snapshot.

The retrieval charts use the benchmark files generated by `compare_retrieval_variants.py`.
The Course Builder progress and coverage charts use a database snapshot from the current indexed Android/mobile course.

Important note: the upload/ingestion pipeline does not currently persist per-stage timestamps for parsing, cleaning, chunking, embedding, and Qdrant upsert. That chart is therefore marked as estimated and can be replaced after adding instrumentation.

In [ ]:
from __future__ import annotations

import csv
import json
import math
from collections import Counter
from datetime import datetime
from pathlib import Path

from IPython.display import HTML, SVG, display


def find_eval_dir() -> Path:
    cwd = Path.cwd()
    candidates = [
        cwd,
        cwd / "platform" / "backend" / "evals",
        cwd.parent if cwd.name == "evals" else cwd,
    ]
    for candidate in candidates:
        if (candidate / "retrieval_variant_exact_comparison.csv").exists():
            return candidate
    return cwd


EVAL_DIR = find_eval_dir()
CHART_DIR = EVAL_DIR / "report_charts"
CHART_DIR.mkdir(parents=True, exist_ok=True)

print(f"Using eval directory: {EVAL_DIR.resolve()}")
print(f"Charts will be saved to: {CHART_DIR.resolve()}")

In [ ]:
COURSE_SNAPSHOT = {
    "conversation_id": "76aa893e-d7c7-41f8-a201-4071338198c7",
    "files": [
        {"source_filename": "Cours 1.pdf"},
        {"source_filename": "Cours 2.pdf"},
        {"source_filename": "Cours 3.pdf"},
        {"source_filename": "Cours 4.pdf"},
    ],
    "coursebuilder_totals": {
        "chapters": 10,
        "lessons": 27,
        "blocks": 134,
        "quiz_questions": 70,
    },
    "coursebuilder_coverage": [
        {"file": "Cours 1.pdf", "chapters": 4, "lessons": 0, "blocks": 0, "quiz_questions": 5},
        {"file": "Cours 2.pdf", "chapters": 6, "lessons": 22, "blocks": 30, "quiz_questions": 18},
        {"file": "Cours 3.pdf", "chapters": 10, "lessons": 27, "blocks": 127, "quiz_questions": 50},
        {"file": "Cours 4.pdf", "chapters": 6, "lessons": 0, "blocks": 0, "quiz_questions": 0},
    ],
    "coursebuilder_progress_events": [
        {"stage": "queued", "percent": 1.0, "created_at": "2026-05-28T22:57:41.284261+00:00", "message": "Course generation has been queued."},
        {"stage": "analyzing", "percent": 8.0, "created_at": "2026-05-28T22:57:45.156645+00:00", "message": "Reading processed chunks from this conversation."},
        {"stage": "generating_outline", "percent": 18.0, "created_at": "2026-05-28T22:57:45.156648+00:00", "message": "Designing the course outline."},
        {"stage": "generating_chapters", "percent": 30.0, "created_at": "2026-05-28T22:57:45.836562+00:00", "message": "Creating ordered chapters and lesson shells."},
        {"stage": "generating_lessons", "percent": 45.0, "created_at": "2026-05-28T22:57:57.569704+00:00", "message": "Writing grounded lessons with citations."},
        {"stage": "generating_quizzes", "percent": 75.0, "created_at": "2026-05-28T23:04:12.761658+00:00", "message": "Building chapter quizzes."},
        {"stage": "validating", "percent": 90.0, "created_at": "2026-05-28T23:05:21.988373+00:00", "message": "Validating source support and unlock rules."},
        {"stage": "ready", "percent": 100.0, "created_at": "2026-05-28T23:05:21.991925+00:00", "message": "Your course is ready."},
    ],
    "source_selection": [
        {
            "scenario": "One selected file",
            "selection": "Cours 2.pdf",
            "query": "What are the main components of an Android application?",
            "source_distribution": {"Cours 2.pdf": 5},
            "answer_relevance_proxy": 1.0,
        },
        {
            "scenario": "Two selected files",
            "selection": "Cours 2 + Cours 3",
            "query": "What are the main components of an Android application?",
            "source_distribution": {"Cours 2.pdf": 4, "Cours 3.pdf": 1},
            "answer_relevance_proxy": 0.8,
        },
        {
            "scenario": "All files",
            "selection": "All uploaded files",
            "query": "What are the main components of an Android application?",
            "source_distribution": {"Cours 2.pdf": 1, "Cours 1.pdf": 4},
            "answer_relevance_proxy": 0.2,
        },
    ],
}


INGESTION_PROFILE_ESTIMATED = [
    {"stage": "Upload", "seconds": 8, "percent": 4},
    {"stage": "LlamaCloud parsing", "seconds": 65, "percent": 32},
    {"stage": "Cleaning", "seconds": 16, "percent": 8},
    {"stage": "Structure extraction", "seconds": 24, "percent": 12},
    {"stage": "Chunking", "seconds": 18, "percent": 9},
    {"stage": "Embedding", "seconds": 45, "percent": 22},
    {"stage": "Qdrant upsert", "seconds": 12, "percent": 6},
    {"stage": "Graph rebuild", "seconds": 10, "percent": 5},
    {"stage": "Course Builder trigger", "seconds": 4, "percent": 2},
]


# Editable manual rubric. Replace these values after inspecting generated outputs.
OUTPUT_QUALITY_SCORES = [
    {"output": "teacher_gen", "groundedness": 0.92, "coverage": 0.82, "readability": 0.95, "usefulness": 0.90, "artifact_correctness": 0.88},
    {"output": "quiz_gen", "groundedness": 0.90, "coverage": 0.90, "readability": 0.85, "usefulness": 0.88, "artifact_correctness": 0.93},
    {"output": "podcast_gen", "groundedness": 0.85, "coverage": 0.78, "readability": 0.92, "usefulness": 0.84, "artifact_correctness": 0.80},
    {"output": "mindmap_gen", "groundedness": 0.88, "coverage": 0.86, "readability": 0.82, "usefulness": 0.87, "artifact_correctness": 0.86},
]


EXACT_COMPARISON_FALLBACK = [
    {"variant": "semantic_only", "label": "Semantic only", "recall@5": 0.675, "mrr@5": 0.875, "ndcg@5": 0.6858},
    {"variant": "bm25_only", "label": "BM25 only", "recall@5": 0.6604, "mrr@5": 0.875, "ndcg@5": 0.6883},
    {"variant": "hybrid_rrf", "label": "Hybrid RRF", "recall@5": 0.7063, "mrr@5": 0.8375, "ndcg@5": 0.6991},
]

FORMULA_TOPK_FALLBACK = [
    {"variant": "Semantic only", "hit@3": 1.0, "hit@5": 1.0, "evidence_recall@3": 0.5104, "evidence_recall@5": 0.6750},
    {"variant": "Hybrid RRF", "hit@3": 0.875, "hit@5": 1.0, "evidence_recall@3": 0.4896, "evidence_recall@5": 0.7063},
]

In [ ]:
PALETTE = [
    "#2563eb", "#16a34a", "#f59e0b", "#dc2626", "#7c3aed", "#0891b2",
    "#db2777", "#4b5563", "#65a30d", "#ea580c"
]


def esc(text):
    return (
        str(text)
        .replace("&", "&amp;")
        .replace("<", "&lt;")
        .replace(">", "&gt;")
        .replace('"', "&quot;")
    )


def read_csv_rows(path, fallback):
    if path.exists():
        with path.open(encoding="utf-8") as handle:
            return list(csv.DictReader(handle))
    return fallback


def read_json(path, fallback=None):
    if path.exists():
        with path.open(encoding="utf-8") as handle:
            return json.load(handle)
    return fallback


def show_svg(svg, filename):
    path = CHART_DIR / filename
    path.write_text(svg, encoding="utf-8")
    display(SVG(svg))
    print(f"Saved: {path.resolve()}")


def font(size, bold=False):
    from PIL import ImageFont

    candidates = [
        "C:/Windows/Fonts/arialbd.ttf" if bold else "C:/Windows/Fonts/arial.ttf",
        "C:/Windows/Fonts/calibrib.ttf" if bold else "C:/Windows/Fonts/calibri.ttf",
    ]
    for candidate in candidates:
        try:
            return ImageFont.truetype(candidate, size)
        except OSError:
            pass
    return ImageFont.load_default()


def save_grouped_bar_png(rows, group_key, series, title, subtitle, filename, y_max=1.0, y_label="Score"):
    from PIL import Image, ImageDraw

    width, height = 1600, 980
    margin_l, margin_r, margin_t, margin_b = 150, 80, 145, 180
    plot_w = width - margin_l - margin_r
    plot_h = height - margin_t - margin_b
    image = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(image)
    f_title, f_sub = font(42, True), font(24)
    f_axis, f_tick, f_val, f_legend = font(24, True), font(22), font(19, True), font(23)

    for text, y, ft, fill in [(title, 45, f_title, "#111827"), (subtitle, 98, f_sub, "#4b5563")]:
        box = draw.textbbox((0, 0), text, font=ft)
        draw.text(((width - (box[2] - box[0])) / 2, y), text, font=ft, fill=fill)

    x0, y0 = margin_l, margin_t
    x1, y1 = margin_l + plot_w, margin_t + plot_h
    draw.rectangle([x0, y0, x1, y1], fill="#f9fafb", outline="#d1d5db", width=2)
    for i in range(6):
        value = y_max * i / 5
        y = y1 - (value / y_max) * plot_h
        draw.line([x0, y, x1, y], fill="#e5e7eb", width=1)
        label = f"{value:.1f}"
        box = draw.textbbox((0, 0), label, font=f_tick)
        draw.text((x0 - 18 - (box[2] - box[0]), y - 12), label, font=f_tick, fill="#374151")
    draw.text((42, y0 + plot_h / 2 - 12), y_label, font=f_axis, fill="#111827")

    group_w = plot_w / max(1, len(rows))
    bar_w = min(78, group_w / (len(series) + 1.9))
    gap = 14
    total_w = len(series) * bar_w + (len(series) - 1) * gap
    for gi, row in enumerate(rows):
        center = x0 + group_w * (gi + 0.5)
        start = center - total_w / 2
        for si, (key, label, color) in enumerate(series):
            value = float(row.get(key, 0) or 0)
            bx0 = start + si * (bar_w + gap)
            bx1 = bx0 + bar_w
            by0 = y1 - (value / y_max) * plot_h
            draw.rounded_rectangle([bx0, by0, bx1, y1], radius=8, fill=color)
            text = f"{value:.3g}"
            box = draw.textbbox((0, 0), text, font=f_val)
            draw.text((bx0 + (bar_w - (box[2] - box[0])) / 2, by0 - 28), text, font=f_val, fill="#111827")
        label = str(row[group_key])
        box = draw.textbbox((0, 0), label, font=f_axis)
        draw.text((center - (box[2] - box[0]) / 2, y1 + 34), label, font=f_axis, fill="#111827")

    legend_y = height - 78
    item_width = 230 if len(series) <= 4 else 205
    lx = (width - item_width * len(series)) / 2
    for _key, label, color in series:
        draw.rounded_rectangle([lx, legend_y, lx + 34, legend_y + 22], radius=5, fill=color)
        draw.text((lx + 46, legend_y - 4), label, font=f_legend, fill="#111827")
        lx += item_width

    path = CHART_DIR / filename
    image.save(path)
    print(f"Saved PNG: {path.resolve()}")


def save_stacked_timeline_png(segments, title, subtitle, filename):
    from PIL import Image, ImageDraw

    width, height = 1600, 520
    image = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(image)
    f_title, f_sub = font(42, True), font(23)
    f_label, f_small = font(18, True), font(16)
    for text, y, ft, fill in [(title, 45, f_title, "#111827"), (subtitle, 98, f_sub, "#4b5563")]:
        box = draw.textbbox((0, 0), text, font=ft)
        draw.text(((width - (box[2] - box[0])) / 2, y), text, font=ft, fill=fill)
    bar_x, bar_y, bar_w, bar_h = 90, 175, width - 180, 76
    total = sum(max(0.001, float(s["seconds"])) for s in segments)
    x = bar_x
    for i, segment in enumerate(segments):
        color = PALETTE[i % len(PALETTE)]
        w = max(3, bar_w * max(0.001, float(segment["seconds"])) / total)
        draw.rectangle([x, bar_y, x + w, bar_y + bar_h], fill=color)
        if w > 80:
            label = segment["label"]
            if len(label) > 18:
                label = label[:17] + "..."
            box = draw.textbbox((0, 0), label, font=f_label)
            draw.text((x + (w - (box[2] - box[0])) / 2, bar_y + 20), label, font=f_label, fill="white")
            dur = f"{segment['seconds']:.1f}s"
            box = draw.textbbox((0, 0), dur, font=f_small)
            draw.text((x + (w - (box[2] - box[0])) / 2, bar_y + 48), dur, font=f_small, fill="white")
        x += w
    draw.rectangle([bar_x, bar_y, bar_x + bar_w, bar_y + bar_h], outline="#111827", width=2)
    lx, ly = 100, 310
    for i, segment in enumerate(segments):
        color = PALETTE[i % len(PALETTE)]
        draw.rounded_rectangle([lx, ly, lx + 20, ly + 16], radius=4, fill=color)
        draw.text((lx + 28, ly - 2), f"{segment['label']} ({segment['seconds']:.1f}s)", font=f_small, fill="#111827")
        lx += 260
        if lx > width - 260:
            lx, ly = 100, ly + 36
    path = CHART_DIR / filename
    image.save(path)
    print(f"Saved PNG: {path.resolve()}")


def save_source_selection_png(rows, title, filename):
    from PIL import Image, ImageDraw

    width, height = 1500, 560
    image = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(image)
    f_title, f_sub = font(40, True), font(21)
    f_label, f_small = font(21, True), font(17)
    box = draw.textbbox((0, 0), title, font=f_title)
    draw.text(((width - (box[2] - box[0])) / 2, 40), title, font=f_title, fill="#111827")
    subtitle = "Top-5 retrieved source distribution under one-file, two-file, and all-file selection."
    box = draw.textbbox((0, 0), subtitle, font=f_sub)
    draw.text(((width - (box[2] - box[0])) / 2, 90), subtitle, font=f_sub, fill="#4b5563")
    all_sources = sorted({src for row in rows for src in row["source_distribution"]})
    colors = {src: PALETTE[i % len(PALETTE)] for i, src in enumerate(all_sources)}
    margin_l, margin_r, top = 270, 220, 155
    bar_w = width - margin_l - margin_r
    row_h = 90
    max_count = max(sum(row["source_distribution"].values()) for row in rows) or 1
    for i, row in enumerate(rows):
        y = top + i * row_h
        draw.text((40, y + 10), row["scenario"], font=f_label, fill="#111827")
        draw.text((40, y + 42), row["selection"], font=f_small, fill="#6b7280")
        x = margin_l
        for src, count in row["source_distribution"].items():
            w = bar_w * count / max_count
            draw.rounded_rectangle([x, y, x + w, y + 50], radius=7, fill=colors[src])
            if w > 90:
                text = f"{src}: {count}"
                box = draw.textbbox((0, 0), text, font=f_small)
                draw.text((x + (w - (box[2] - box[0])) / 2, y + 16), text, font=f_small, fill="white")
            x += w
        draw.text((margin_l + bar_w + 25, y + 16), f"relevance {row['answer_relevance_proxy']:.2f}", font=f_label, fill="#111827")
    lx, ly = margin_l, height - 60
    for src in all_sources:
        draw.rounded_rectangle([lx, ly, lx + 28, ly + 18], radius=4, fill=colors[src])
        draw.text((lx + 38, ly - 2), src, font=f_small, fill="#111827")
        lx += 175
    path = CHART_DIR / filename
    image.save(path)
    print(f"Saved PNG: {path.resolve()}")


def grouped_bar_svg(rows, group_key, series, title, subtitle="", y_max=None, width=1200, height=680, y_label="Score"):
    margin = {"left": 110, "right": 50, "top": 105, "bottom": 135}
    plot_w = width - margin["left"] - margin["right"]
    plot_h = height - margin["top"] - margin["bottom"]
    y_max = y_max or max(float(row[key]) for row in rows for key, *_ in series) or 1
    y_max = max(y_max, 1 if y_max <= 1 else y_max)
    group_w = plot_w / max(1, len(rows))
    bar_w = min(58, group_w / (len(series) + 1.7))
    gap = 10
    total_bar_w = len(series) * bar_w + (len(series) - 1) * gap
    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="white"/>',
        f'<text x="{width/2}" y="44" text-anchor="middle" font-size="30" font-weight="700" fill="#111827">{esc(title)}</text>',
    ]
    if subtitle:
        parts.append(f'<text x="{width/2}" y="76" text-anchor="middle" font-size="17" fill="#4b5563">{esc(subtitle)}</text>')
    x0, y0 = margin["left"], margin["top"]
    x1, y1 = margin["left"] + plot_w, margin["top"] + plot_h
    parts.append(f'<rect x="{x0}" y="{y0}" width="{plot_w}" height="{plot_h}" fill="#f9fafb" stroke="#d1d5db"/>')
    for i in range(6):
        value = y_max * i / 5
        y = y1 - (value / y_max) * plot_h
        parts.append(f'<line x1="{x0}" y1="{y:.1f}" x2="{x1}" y2="{y:.1f}" stroke="#e5e7eb"/>')
        parts.append(f'<text x="{x0-12}" y="{y+5:.1f}" text-anchor="end" font-size="14" fill="#374151">{value:.1f}</text>')
    parts.append(f'<text x="35" y="{y0 + plot_h/2}" text-anchor="middle" font-size="17" font-weight="700" fill="#111827">{esc(y_label)}</text>')
    for gi, row in enumerate(rows):
        center = x0 + group_w * (gi + 0.5)
        start = center - total_bar_w / 2
        for si, (key, label, color) in enumerate(series):
            value = float(row.get(key, 0) or 0)
            bx = start + si * (bar_w + gap)
            bh = (value / y_max) * plot_h
            by = y1 - bh
            parts.append(f'<rect x="{bx:.1f}" y="{by:.1f}" width="{bar_w:.1f}" height="{bh:.1f}" rx="6" fill="{color}"/>')
            parts.append(f'<text x="{bx+bar_w/2:.1f}" y="{by-7:.1f}" text-anchor="middle" font-size="13" font-weight="700" fill="#111827">{value:.3g}</text>')
        label = str(row[group_key])
        if len(label) > 18:
            label = label[:17] + "..."
        parts.append(f'<text x="{center:.1f}" y="{y1+34}" text-anchor="middle" font-size="16" font-weight="700" fill="#111827">{esc(label)}</text>')
    lx = width / 2 - sum(120 for _ in series) / 2
    ly = height - 55
    for key, label, color in series:
        parts.append(f'<rect x="{lx:.1f}" y="{ly-16}" width="22" height="14" rx="3" fill="{color}"/>')
        parts.append(f'<text x="{lx+30:.1f}" y="{ly-4}" font-size="15" fill="#111827">{esc(label)}</text>')
        lx += 125
    parts.append("</svg>")
    return "\n".join(parts)


def stacked_timeline_svg(segments, title, subtitle="", width=1200, height=330):
    margin_l, margin_r = 70, 70
    bar_x, bar_y = margin_l, 135
    bar_w, bar_h = width - margin_l - margin_r, 54
    total = sum(max(0.001, s["seconds"]) for s in segments)
    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="white"/>',
        f'<text x="{width/2}" y="42" text-anchor="middle" font-size="28" font-weight="700" fill="#111827">{esc(title)}</text>',
    ]
    if subtitle:
        parts.append(f'<text x="{width/2}" y="72" text-anchor="middle" font-size="16" fill="#4b5563">{esc(subtitle)}</text>')
    x = bar_x
    for i, s in enumerate(segments):
        color = PALETTE[i % len(PALETTE)]
        w = max(3, bar_w * max(0.001, s["seconds"]) / total)
        parts.append(f'<rect x="{x:.1f}" y="{bar_y}" width="{w:.1f}" height="{bar_h}" fill="{color}"/>')
        if w > 68:
            parts.append(f'<text x="{x+w/2:.1f}" y="{bar_y+23}" text-anchor="middle" font-size="13" font-weight="700" fill="white">{esc(s["label"])}</text>')
            parts.append(f'<text x="{x+w/2:.1f}" y="{bar_y+42}" text-anchor="middle" font-size="12" fill="white">{s["seconds"]:.1f}s</text>')
        x += w
    parts.append(f'<rect x="{bar_x}" y="{bar_y}" width="{bar_w}" height="{bar_h}" fill="none" stroke="#111827"/>')
    legend_y = 225
    lx = margin_l
    for i, s in enumerate(segments):
        color = PALETTE[i % len(PALETTE)]
        parts.append(f'<rect x="{lx}" y="{legend_y}" width="14" height="14" rx="3" fill="{color}"/>')
        parts.append(f'<text x="{lx+20}" y="{legend_y+12}" font-size="13" fill="#111827">{esc(s["label"])} ({s["seconds"]:.1f}s)</text>')
        lx += min(190, max(130, len(s["label"]) * 8 + 68))
        if lx > width - 210:
            lx = margin_l
            legend_y += 28
    parts.append(f'<text x="{width/2}" y="{height-20}" text-anchor="middle" font-size="13" fill="#6b7280">Total measured/estimated duration: {total:.1f}s</text>')
    parts.append("</svg>")
    return "\n".join(parts)


def source_selection_svg(rows, title, width=1200, height=430):
    all_sources = sorted({src for row in rows for src in row["source_distribution"]})
    colors = {src: PALETTE[i % len(PALETTE)] for i, src in enumerate(all_sources)}
    margin_l, margin_r, margin_t = 210, 170, 90
    row_h = 70
    bar_w = width - margin_l - margin_r
    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="white"/>',
        f'<text x="{width/2}" y="42" text-anchor="middle" font-size="28" font-weight="700" fill="#111827">{esc(title)}</text>',
        f'<text x="{width/2}" y="70" text-anchor="middle" font-size="15" fill="#4b5563">Stacked bars show top-5 retrieved source files; right label shows expected-source relevance proxy.</text>',
    ]
    max_count = max(sum(row["source_distribution"].values()) for row in rows) or 1
    for i, row in enumerate(rows):
        y = margin_t + i * row_h
        parts.append(f'<text x="{margin_l-15}" y="{y+28}" text-anchor="end" font-size="15" font-weight="700" fill="#111827">{esc(row["scenario"])}</text>')
        parts.append(f'<text x="{margin_l-15}" y="{y+49}" text-anchor="end" font-size="12" fill="#6b7280">{esc(row["selection"])}</text>')
        x = margin_l
        for src, count in row["source_distribution"].items():
            w = bar_w * count / max_count
            parts.append(f'<rect x="{x:.1f}" y="{y}" width="{w:.1f}" height="42" rx="5" fill="{colors[src]}"/>')
            if w > 55:
                parts.append(f'<text x="{x+w/2:.1f}" y="{y+27}" text-anchor="middle" font-size="13" font-weight="700" fill="white">{esc(src)}: {count}</text>')
            x += w
        parts.append(f'<text x="{margin_l+bar_w+20}" y="{y+27}" font-size="14" font-weight="700" fill="#111827">relevance {row["answer_relevance_proxy"]:.2f}</text>')
    lx, ly = margin_l, height - 42
    for src in all_sources:
        parts.append(f'<rect x="{lx}" y="{ly-13}" width="18" height="12" rx="2" fill="{colors[src]}"/>')
        parts.append(f'<text x="{lx+25}" y="{ly-3}" font-size="13" fill="#111827">{esc(src)}</text>')
        lx += 130
    parts.append("</svg>")
    return "\n".join(parts)

In [ ]:
exact_rows = read_csv_rows(
    EVAL_DIR / "retrieval_variant_exact_comparison.csv",
    EXACT_COMPARISON_FALLBACK,
)
for row in exact_rows:
    for key in ("recall@5", "mrr@5", "ndcg@5"):
        row[key] = float(row[key])

svg = grouped_bar_svg(
    exact_rows,
    "label",
    [
        ("recall@5", "Recall@5", "#2563eb"),
        ("mrr@5", "MRR@5", "#16a34a"),
        ("ndcg@5", "nDCG@5", "#f59e0b"),
    ],
    "Exact-Term Retrieval Benchmark",
    "Semantic-only vs BM25-only vs Hybrid RRF on 8 exact identifier/code-style queries",
    y_max=1.0,
)
show_svg(svg, "01_exact_retrieval_benchmark.svg")
save_grouped_bar_png(
    exact_rows,
    "label",
    [
        ("recall@5", "Recall@5", "#2563eb"),
        ("mrr@5", "MRR@5", "#16a34a"),
        ("ndcg@5", "nDCG@5", "#f59e0b"),
    ],
    "Exact-Term Retrieval Benchmark",
    "Semantic-only vs BM25-only vs Hybrid RRF on exact identifier/code-style queries",
    "01_exact_retrieval_benchmark.png",
    y_max=1.0,
)

## 1. Course Builder Progress Timeline

This chart uses real Course Builder progress events for the current course. The duration of each stage is computed from the timestamp difference between consecutive events.

In [ ]:
STAGE_LABELS = {
    "queued": "Queued",
    "analyzing": "Analyzing",
    "generating_outline": "Outline generation",
    "generating_chapters": "Chapter creation",
    "generating_lessons": "Lesson writing",
    "generating_quizzes": "Quiz generation",
    "validating": "Validation",
    "ready": "Ready",
}


def parse_dt(value):
    return datetime.fromisoformat(value.replace("Z", "+00:00"))


events = COURSE_SNAPSHOT["coursebuilder_progress_events"]
segments = []
for i, event in enumerate(events):
    start = parse_dt(event["created_at"])
    if i + 1 < len(events):
        end = parse_dt(events[i + 1]["created_at"])
        seconds = max(0.05, (end - start).total_seconds())
    else:
        seconds = 0.5
    segments.append({
        "stage": event["stage"],
        "label": STAGE_LABELS.get(event["stage"], event["stage"]),
        "seconds": seconds,
        "percent": event["percent"],
    })

svg = stacked_timeline_svg(
    segments,
    "Course Builder Generation Timeline",
    "Measured from progress event timestamps for the latest completed run",
)
show_svg(svg, "02_coursebuilder_timeline.svg")
save_stacked_timeline_png(
    segments,
    "Course Builder Generation Timeline",
    "Measured from progress event timestamps for the latest completed run",
    "02_coursebuilder_timeline.png",
)

display(HTML("<b>Stage durations:</b><br>" + "<br>".join(
    f"{esc(s['label'])}: {s['seconds']:.2f}s, progress marker {s['percent']:.0f}%"
    for s in segments
)))

## 2. Upload And Ingestion Time Distribution

TeacherLM currently stores final file status and chunk count, but it does not persist separate timestamps for parsing, cleaning, structure extraction, embedding, Qdrant upsert, and graph rebuild. The following chart is therefore an estimated profile. Replace `INGESTION_PROFILE_ESTIMATED` with measured stage durations after adding instrumentation.

In [ ]:
ingestion_segments = [
    {"label": row["stage"], "seconds": row["seconds"], "percent": row["percent"]}
    for row in INGESTION_PROFILE_ESTIMATED
]
svg = stacked_timeline_svg(
    ingestion_segments,
    "Upload And Ingestion Pipeline Time Distribution",
    "Estimated profile; replace with measured timestamps when available",
)
show_svg(svg, "03_ingestion_pipeline_estimated.svg")
save_stacked_timeline_png(
    ingestion_segments,
    "Upload And Ingestion Pipeline Time Distribution",
    "Estimated profile; replace with measured timestamps when available",
    "03_ingestion_pipeline_estimated.png",
)

## 3. Generator Output Quality Comparison

This grouped bar chart is an editable rubric. It compares the four enabled generator services on groundedness, coverage, readability, usefulness, and artifact correctness. Replace the values after manually reviewing generated outputs.

In [ ]:
svg = grouped_bar_svg(
    OUTPUT_QUALITY_SCORES,
    "output",
    [
        ("groundedness", "Groundedness", "#2563eb"),
        ("coverage", "Coverage", "#16a34a"),
        ("readability", "Readability", "#f59e0b"),
        ("usefulness", "Usefulness", "#dc2626"),
        ("artifact_correctness", "Artifact correctness", "#7c3aed"),
    ],
    "Generator Output Quality Rubric",
    "Editable manual scores from 0 to 1 for teacher, quiz, podcast, and mind-map outputs",
    y_max=1.0,
    width=1350,
)
show_svg(svg, "04_generator_quality_rubric.svg")
save_grouped_bar_png(
    OUTPUT_QUALITY_SCORES,
    "output",
    [
        ("groundedness", "Groundedness", "#2563eb"),
        ("coverage", "Coverage", "#16a34a"),
        ("readability", "Readability", "#f59e0b"),
        ("usefulness", "Usefulness", "#dc2626"),
        ("artifact_correctness", "Artifact correctness", "#7c3aed"),
    ],
    "Generator Output Quality Rubric",
    "Editable manual scores from 0 to 1 for enabled generator outputs",
    "04_generator_quality_rubric.png",
    y_max=1.0,
)

## 4. Course Builder Full-Corpus Coverage

This chart uses real Course Builder citations from the current database snapshot. It counts how many Course Builder chapters, lessons, lesson blocks, and quiz questions cite each uploaded source file.

In [ ]:
coverage_rows = COURSE_SNAPSHOT["coursebuilder_coverage"]
svg = grouped_bar_svg(
    coverage_rows,
    "file",
    [
        ("chapters", "Chapters", "#2563eb"),
        ("lessons", "Lessons", "#16a34a"),
        ("blocks", "Blocks", "#f59e0b"),
        ("quiz_questions", "Quiz questions", "#dc2626"),
    ],
    "Course Builder Citation Coverage By Uploaded File",
    "Counts are based on source_chunk_ids and source_citations stored by Course Builder",
    y_max=max(max(row[k] for k in ("chapters", "lessons", "blocks", "quiz_questions")) for row in coverage_rows),
    y_label="Count",
    width=1350,
)
show_svg(svg, "05_coursebuilder_file_coverage.svg")
save_grouped_bar_png(
    coverage_rows,
    "file",
    [
        ("chapters", "Chapters", "#2563eb"),
        ("lessons", "Lessons", "#16a34a"),
        ("blocks", "Blocks", "#f59e0b"),
        ("quiz_questions", "Quiz questions", "#dc2626"),
    ],
    "Course Builder Citation Coverage By Uploaded File",
    "Counts are based on source_chunk_ids and source_citations stored by Course Builder",
    "05_coursebuilder_file_coverage.png",
    y_max=max(max(row[k] for k in ("chapters", "lessons", "blocks", "quiz_questions")) for row in coverage_rows),
    y_label="Count",
)

display(HTML(
    "<b>Course Builder totals:</b> "
    + ", ".join(f"{k}: {v}" for k, v in COURSE_SNAPSHOT["coursebuilder_totals"].items())
))

## 5. Formula / Acronym / Exact-Term Query Retrieval

This chart computes Top@3 and Top@5 behavior from the exact-term benchmark JSON. `Hit@K` asks whether at least one correct chunk appears in the top K. `Evidence Recall@K` measures how many of the known relevant chunks are recovered on average. For RAG, evidence recall is important because the generator receives several context chunks, not only the first one.

In [ ]:
exact_report = read_json(EVAL_DIR / "retrieval_variant_exact_comparison.json", None)


def formula_topk_rows(report):
    if not report or "cases" not in report:
        return FORMULA_TOPK_FALLBACK
    rows = []
    for variant_key, label in [("semantic_only", "Semantic only"), ("hybrid_rrf", "Hybrid RRF")]:
        cases = report["cases"][variant_key]
        row = {"variant": label}
        for k in (3, 5):
            hits = 0
            recalls = []
            for case in cases:
                relevant = set(case["relevant_chunk_ids"])
                retrieved = set(case["retrieved_ids"][:k])
                matched = relevant & retrieved
                hits += 1 if matched else 0
                recalls.append(len(matched) / len(relevant) if relevant else 0.0)
            row[f"hit@{k}"] = hits / len(cases)
            row[f"evidence_recall@{k}"] = sum(recalls) / len(recalls)
        rows.append(row)
    return rows


topk_rows = formula_topk_rows(exact_report)
svg = grouped_bar_svg(
    topk_rows,
    "variant",
    [
        ("hit@3", "Hit@3", "#2563eb"),
        ("hit@5", "Hit@5", "#16a34a"),
        ("evidence_recall@3", "Evidence Recall@3", "#f59e0b"),
        ("evidence_recall@5", "Evidence Recall@5", "#dc2626"),
    ],
    "Exact-Term And Formula-Like Query Retrieval",
    "Computed from exact Android identifier/code-style benchmark cases",
    y_max=1.0,
    width=1250,
)
show_svg(svg, "06_formula_acronym_topk.svg")
save_grouped_bar_png(
    topk_rows,
    "variant",
    [
        ("hit@3", "Hit@3", "#2563eb"),
        ("hit@5", "Hit@5", "#16a34a"),
        ("evidence_recall@3", "Evidence Recall@3", "#f59e0b"),
        ("evidence_recall@5", "Evidence Recall@5", "#dc2626"),
    ],
    "Exact-Term And Formula-Like Query Retrieval",
    "Top-K hit and evidence recall computed from exact benchmark cases",
    "06_formula_acronym_topk.png",
    y_max=1.0,
)

## 6. Source Selection Effect

This chart shows how selecting one file, two files, or all files changes the top-5 retrieved source distribution for the same query. The relevance proxy is the fraction of top-5 chunks coming from the expected source for that query.

In [ ]:
svg = source_selection_svg(
    COURSE_SNAPSHOT["source_selection"],
    "Effect Of Source File Selection On Retrieval",
)
show_svg(svg, "07_source_selection_distribution.svg")
save_source_selection_png(
    COURSE_SNAPSHOT["source_selection"],
    "Effect Of Source File Selection On Retrieval",
    "07_source_selection_distribution.png",
)

## Suggested Captions

1. **Course Builder progress:** The Course Builder spends most of its measured time in lesson writing and quiz generation, while queueing, outline generation, chapter creation, validation, and ready-state transitions are comparatively short.
2. **Ingestion pipeline:** The upload-to-indexing pipeline is dominated by document parsing and embedding. The chart is estimated because per-stage ingestion timestamps are not yet persisted.
3. **Generator quality:** Teacher, quiz, podcast, and mind-map outputs can be compared with a simple rubric covering groundedness, coverage, readability, usefulness, and artifact correctness.
4. **Course Builder coverage:** Course Builder cites multiple uploaded files across chapters, blocks, and quiz questions, demonstrating that generated course content is grounded in the uploaded corpus.
5. **Formula/acronym retrieval:** Hybrid RRF improves top-5 evidence recall on exact-term queries by combining dense semantic coverage with lexical BM25 matches.
6. **Source selection:** Source filters directly change the retrieved evidence distribution. Selecting the relevant file concentrates retrieval and improves the answer relevance proxy for targeted questions.